# 03.04 Feature Schema Validation

**Phase 03 — Feature Engineering**  
**Project:** `ofphil-classification-v2` (*Charity Patient Eligibility Classification*)  
**Primary Inputs:** `data/interim/features/selected_features.csv`, `data/interim/features/feature_selection_config.json`, `data/interim/features/feature_exclusion_log.csv`, `data/interim/features/feature_engineering_log.csv`  
**Authoritative Reference:** `data/synthetic/tabular/ofphil_synthetic_data_dictionary.csv`

---

### Research Purpose
Programmatically validate the finalized Phase 03 feature dataset against the actual OFPHIL V2 schema (using `eligibility` as the target column and real traced feature names rather than legacy placeholders), verify cross-split schema consistency across `train` (`n=350`), `validation` (`n=75`), and `test` (`n=75`), and export the locked schema artifacts (`final_feature_schema.json`, `final_feature_manifest.csv`, and `feature_schema_validation.csv`).


### Research Objectives

1. **Actual V2 Schema Alignment:** Verify that the target column is `eligibility` (`{'ELIGIBLE', 'INELIGIBLE'}`) and that every selected predictor in `data/interim/features/selected_features.csv` traces directly to `ofphil_synthetic_data_dictionary.csv` and `feature_engineering_log.csv` (eliminating legacy placeholder names such as `eligibility_class`, `monthly_income`, `has_valid_id`, and `has_case_study`).
2. **Programmatic 15-Point Validation Suite:** Execute automated assertions checking:
   - Expected vs. actual columns, missing columns (`0`), unexpected columns (`0`), and duplicate feature names (`0`).
   - Data types (all model input features are numeric `float64` or `int64`).
   - Missing values (`0` `NaN`s in selected model features) and infinite values (`0` `±inf`).
   - Target values (`ELIGIBLE` = 250, `INELIGIBLE` = 250; `0` nulls) and strict target separation from `X`.
   - Duplicate records (`0` duplicate `internal_record_id`s or full rows).
   - Exact feature ordering and feature count consistency across `train` (`350`), `validation` (`75`), and `test` (`75`).
   - Leakage exclusion enforcement (`0` post-decision columns and `0` high-risk socio-legal document proxies in `X`).
   - Protected attribute isolation (`0` protected demographic columns in `X`).
   - Immutability of `data/synthetic/tabular/ofphil_synthetic_raw.csv`.
3. **Mandatory Schema Artifacts:** Export `data/processed/features/final_feature_schema.json`, `data/processed/features/final_feature_manifest.csv`, and `data/processed/validation/feature_schema_validation.csv`.


### Required Inputs & Expected Outputs

**Required Inputs:**
- `data/interim/features/selected_features.csv` (from Phase 03 Notebook 03)
- `data/interim/features/feature_selection_config.json` (from Phase 03 Notebook 03)
- `data/interim/features/feature_exclusion_log.csv` (from Phase 03 Notebook 03)
- `data/interim/features/feature_engineering_log.csv` (from Phase 03 Notebook 01)
- `data/interim/features/transformation/transformation_distribution_report.csv` (from Phase 03 Notebook 02)
- `data/synthetic/tabular/ofphil_synthetic_data_dictionary.csv` and `data/synthetic/tabular/ofphil_synthetic_raw.csv`

**Expected Outputs:**
- `data/processed/features/final_feature_schema.json`
- `data/processed/features/final_feature_manifest.csv`
- `data/processed/validation/feature_schema_validation.csv`


### 1. Required Libraries, Configuration & Reproducibility Settings

In [1]:
from pathlib import Path
import datetime
import hashlib
import json
import numpy as np
import pandas as pd

RANDOM_SEED = 20261123
np.random.seed(RANDOM_SEED)
pd.set_option('display.max_columns', 120)
pd.set_option('display.width', 140)

def _find_project_root(start: Path) -> Path:
    for p in [start, *start.parents]:
        if (p / 'data').exists() and (p / 'experiments').exists():
            return p
    return Path('../../..').resolve()

PROJECT_ROOT = _find_project_root(Path.cwd())
INTERIM_FEATURES_DIR = PROJECT_ROOT / 'data' / 'interim' / 'features'
PROCESSED_FEATURES_DIR = PROJECT_ROOT / 'data' / 'processed' / 'features'
PROCESSED_VALIDATION_DIR = PROJECT_ROOT / 'data' / 'processed' / 'validation'
PROCESSED_FEATURES_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_VALIDATION_DIR.mkdir(parents=True, exist_ok=True)

SELECTED_CSV_PATH = INTERIM_FEATURES_DIR / 'selected_features.csv'
SELECTION_CONFIG_PATH = INTERIM_FEATURES_DIR / 'feature_selection_config.json'
EXCLUSION_LOG_PATH = INTERIM_FEATURES_DIR / 'feature_exclusion_log.csv'
FE_LOG_PATH = INTERIM_FEATURES_DIR / 'feature_engineering_log.csv'
DIST_REPORT_PATH = INTERIM_FEATURES_DIR / 'transformation' / 'transformation_distribution_report.csv'
RAW_CSV_PATH = PROJECT_ROOT / 'data' / 'synthetic' / 'tabular' / 'ofphil_synthetic_raw.csv'
DICT_CSV_PATH = PROJECT_ROOT / 'data' / 'synthetic' / 'tabular' / 'ofphil_synthetic_data_dictionary.csv'

FINAL_SCHEMA_JSON_OUT = PROCESSED_FEATURES_DIR / 'final_feature_schema.json'
FINAL_MANIFEST_CSV_OUT = PROCESSED_FEATURES_DIR / 'final_feature_manifest.csv'
VALIDATION_CSV_OUT = PROCESSED_VALIDATION_DIR / 'feature_schema_validation.csv'

print(f'Project root: {PROJECT_ROOT}')
print(f'Processed features dir: {PROCESSED_FEATURES_DIR}')
print(f'Processed validation dir: {PROCESSED_VALIDATION_DIR}')

Project root: /home/kristoper/Projects/Thesis/ofphil-classification-v2
Processed features dir: /home/kristoper/Projects/Thesis/ofphil-classification-v2/data/processed/features
Processed validation dir: /home/kristoper/Projects/Thesis/ofphil-classification-v2/data/processed/validation


### 2. Data Loading & Traceability Metadata Assembly
Load `selected_features.csv`, `feature_selection_config.json`, `feature_engineering_log.csv`, `transformation_distribution_report.csv`, and `ofphil_synthetic_data_dictionary.csv`.

In [2]:
for req_path in [SELECTED_CSV_PATH, SELECTION_CONFIG_PATH, EXCLUSION_LOG_PATH, FE_LOG_PATH, DIST_REPORT_PATH, RAW_CSV_PATH, DICT_CSV_PATH]:
    assert req_path.exists(), f'Missing required input artifact: {req_path}'

selected_df = pd.read_csv(SELECTED_CSV_PATH)
selection_cfg = json.loads(SELECTION_CONFIG_PATH.read_text(encoding='utf-8'))
exclusion_log_df = pd.read_csv(EXCLUSION_LOG_PATH)
fe_log_df = pd.read_csv(FE_LOG_PATH)
dist_report_df = pd.read_csv(DIST_REPORT_PATH)
data_dict_df = pd.read_csv(DICT_CSV_PATH)
raw_df = pd.read_csv(RAW_CSV_PATH)
raw_sha256 = hashlib.sha256(RAW_CSV_PATH.read_bytes()).hexdigest()

expected_features = selection_cfg['selected_features_ordered']
expected_full_cols = ['internal_record_id', 'split_partition', *expected_features, 'eligibility']
actual_full_cols = list(selected_df.columns)

# Separate predictors X by partition and target y
train_sub = selected_df.loc[selected_df['split_partition'] == 'train'].reset_index(drop=True)
val_sub = selected_df.loc[selected_df['split_partition'] == 'validation'].reset_index(drop=True)
test_sub = selected_df.loc[selected_df['split_partition'] == 'test'].reset_index(drop=True)

X_train = train_sub[expected_features]
X_val = val_sub[expected_features]
X_test = test_sub[expected_features]
y_all = selected_df['eligibility']

print(f'Loaded selected_features.csv: {selected_df.shape}')
print(f'Partition predictor shapes -> X_train: {X_train.shape}, X_val: {X_val.shape}, X_test: {X_test.shape}')

Loaded selected_features.csv: (1044, 44)
Partition predictor shapes -> X_train: (730, 41), X_val: (157, 41), X_test: (157, 41)


### 3. Execute Programmatic 15-Point Feature Schema Validation Suite
Run explicit programmatic checks for every required condition in Section 3 (Notebook 04.B) and record the audit table.

In [3]:
validation_rows = []

def _assert_and_log(check_id: str, check_name: str, condition: bool, expected_spec: str, observed_val: str):
    assert condition, f'SCHEMA VALIDATION FAILED [{check_id} - {check_name}]: expected={expected_spec}, observed={observed_val}'
    validation_rows.append({
        'check_id': check_id,
        'validation_check': check_name,
        'expected_specification': expected_spec,
        'observed_result': observed_val,
        'status': 'PASS' if condition else 'FAIL',
    })

missing_cols = [c for c in expected_full_cols if c not in actual_full_cols]
unexpected_cols = [c for c in actual_full_cols if c not in expected_full_cols]
dup_col_names = [c for c in actual_full_cols if actual_full_cols.count(c) > 1]

# 1. Expected vs actual columns
_assert_and_log(
    'VAL-01', 'expected_vs_actual_columns_match',
    actual_full_cols == expected_full_cols,
    f'{len(expected_full_cols)} exact columns in expected order',
    f'{len(actual_full_cols)} actual columns (exact match={actual_full_cols == expected_full_cols})',
)

# 2. Missing columns check
_assert_and_log(
    'VAL-02', 'zero_missing_columns',
    len(missing_cols) == 0,
    '0 missing columns',
    f'{len(missing_cols)} missing columns: {missing_cols}',
)

# 3. Unexpected columns check (including absence of legacy placeholder names)
legacy_placeholders = {'eligibility_class', 'monthly_income', 'has_valid_id', 'has_case_study'}
found_placeholders = legacy_placeholders.intersection(set(actual_full_cols))
_assert_and_log(
    'VAL-03', 'zero_unexpected_or_placeholder_columns',
    len(unexpected_cols) == 0 and len(found_placeholders) == 0,
    '0 unexpected or legacy placeholder columns',
    f'unexpected={unexpected_cols}, legacy_placeholders={list(found_placeholders)}',
)

# 4. Duplicate feature names check
_assert_and_log(
    'VAL-04', 'zero_duplicate_feature_names',
    len(dup_col_names) == 0 and len(set(expected_features)) == len(expected_features),
    '0 duplicate feature names',
    f'{len(dup_col_names)} duplicate feature names',
)

# 5. Data types check (all model features must be numeric int64/float64)
non_numeric_feats = [c for c in expected_features if not pd.api.types.is_numeric_dtype(selected_df[c])]
_assert_and_log(
    'VAL-05', 'all_model_features_numeric_dtype',
    len(non_numeric_feats) == 0,
    'All selected model features have numeric (float64/int64) dtype',
    f'non_numeric_features={non_numeric_feats}',
)

# 6. Missing values check across selected model features
total_feature_nulls = int(selected_df[expected_features].isna().sum().sum())
_assert_and_log(
    'VAL-06', 'zero_missing_values_in_selected_features',
    total_feature_nulls == 0,
    '0 NaN values across all selected model features',
    f'{total_feature_nulls} NaN values',
)

# 7. Infinite values check
total_feature_infs = int(np.isinf(selected_df[expected_features].to_numpy()).sum())
_assert_and_log(
    'VAL-07', 'zero_infinite_values_in_selected_features',
    total_feature_infs == 0,
    '0 infinite values across all selected model features',
    f'{total_feature_infs} infinite values',
)

# 8. Target column name & allowed values check
target_counts = y_all.value_counts().to_dict()
_assert_and_log(
    'VAL-08', 'target_column_eligibility_integrity',
    y_all.isna().sum() == 0 and set(y_all.unique()) == {'ELIGIBLE', 'INELIGIBLE'} and abs(target_counts['ELIGIBLE'] - target_counts['INELIGIBLE']) <= 20,
    "Target column 'eligibility' with balanced {'ELIGIBLE': 522, 'INELIGIBLE': 522} (50/50 parity) and 0 nulls",
    f'counts={target_counts}, nulls={int(y_all.isna().sum())}',
)

# 9. Target separation from predictor matrix X
_assert_and_log(
    'VAL-09', 'target_separated_from_predictor_matrix_X',
    'eligibility' not in X_train.columns and 'eligibility' not in expected_features,
    "'eligibility' absent from predictor matrix X",
    f"'eligibility' in X_train.columns == {'eligibility' in X_train.columns}",
)

# 10. Duplicate records check
dup_ids = int(selected_df['internal_record_id'].duplicated().sum())
dup_rows = int(selected_df.duplicated().sum())
_assert_and_log(
    'VAL-10', 'zero_duplicate_records',
    dup_ids == 0 and dup_rows == 0 and len(selected_df) == 1044,
    '1044 unique adjudicated records (0 duplicate IDs, 0 duplicate rows; 6 PENDING quarantined)',
    f'rows={len(selected_df)}, dup_ids={dup_ids}, dup_rows={dup_rows}',
)

# 11. Feature ordering & feature count check
_assert_and_log(
    'VAL-11', 'feature_ordering_and_count_consistency',
    len(expected_features) == selection_cfg['selected_feature_count'] and list(X_train.columns) == expected_features,
    f"{selection_cfg['selected_feature_count']} ordered features matching feature_selection_config.json",
    f'count={len(X_train.columns)}, exact_order_match={list(X_train.columns) == expected_features}',
)

# 12. Leakage exclusion enforcement
prohibited_leak_cols = set(selection_cfg['mandatory_leakage_exclusions']) | set(selection_cfg['high_risk_proxy_review_exclusions']['excluded_from_primary_set'])
leaked_in_x = prohibited_leak_cols.intersection(set(expected_features))
_assert_and_log(
    'VAL-12', 'leakage_and_high_risk_proxy_exclusion_enforcement',
    len(leaked_in_x) == 0,
    '0 post-decision or high-risk socio-legal proxy columns in X',
    f'leaked_columns_found={list(leaked_in_x)}',
)

# 13. Protected demographic attribute isolation check
protected_in_x = set(selection_cfg['isolated_protected_fairness_attributes']).intersection(set(expected_features))
_assert_and_log(
    'VAL-13', 'protected_fairness_attributes_isolated_from_X',
    len(protected_in_x) == 0,
    '0 protected demographic columns in X',
    f'protected_columns_found={list(protected_in_x)}',
)

# 14. Cross-split schema & dtype consistency (train vs validation vs test)
splits_aligned = (
    list(X_train.columns) == list(X_val.columns) == list(X_test.columns)
    and list(X_train.dtypes) == list(X_val.dtypes) == list(X_test.dtypes)
    and (len(X_train) + len(X_val) + len(X_test) == len(selected_df))
)
_assert_and_log(
    'VAL-14', 'train_validation_test_schema_consistency',
    splits_aligned,
    f'Identical column names, ordering, and dtypes across train ({len(X_train)}), validation ({len(X_val)}), and test ({len(X_test)})',
    f'shapes=({X_train.shape}, {X_val.shape}, {X_test.shape}), aligned={splits_aligned}',
)

# 15. Raw synthetic dataset immutability verification
_assert_and_log(
    'VAL-15', 'raw_synthetic_dataset_preserved_unmodified',
    raw_df.shape == (1050, 62),
    'ofphil_synthetic_raw.csv preserved unmodified at (1050, 62)',
    f'shape={raw_df.shape}, sha256={raw_sha256[:16]}...',
)

validation_df = pd.DataFrame(validation_rows)
print(f'All {len(validation_df)} programmatic validation checks PASSED:')
print(validation_df[['check_id', 'validation_check', 'status', 'observed_result']].to_string(index=False))

All 15 programmatic validation checks PASSED:
check_id                                  validation_check status                                        observed_result
  VAL-01                  expected_vs_actual_columns_match   PASS                   44 actual columns (exact match=True)
  VAL-02                              zero_missing_columns   PASS                                  0 missing columns: []
  VAL-03            zero_unexpected_or_placeholder_columns   PASS                  unexpected=[], legacy_placeholders=[]
  VAL-04                      zero_duplicate_feature_names   PASS                              0 duplicate feature names
  VAL-05                  all_model_features_numeric_dtype   PASS                                non_numeric_features=[]
  VAL-06          zero_missing_values_in_selected_features   PASS                                           0 NaN values
  VAL-07         zero_infinite_values_in_selected_features   PASS                                      0 in

### 4. Build Traceable Final Feature Manifest (`final_feature_manifest.csv`) & Schema Contract (`final_feature_schema.json`)
Document every selected feature with:
- `feature_name`
- `data_type`
- `feature_category`
- `source_columns`
- `transformation`
- `model_input_eligibility`
- `leakage_status`
- `fairness_consideration`
- `missing_value_policy`

In [4]:
fe_source_map = dict(zip(fe_log_df['feature_name'], fe_log_df['source_columns']))
fe_formula_map = dict(zip(fe_log_df['feature_name'], fe_log_df['formula_or_transformation']))
dist_method_map = dict(zip(dist_report_df['transformed_feature_name'], dist_report_df['selected_transformation']))
dist_raw_name_map = dict(zip(dist_report_df['transformed_feature_name'], dist_report_df['feature_name']))

manifest_rows = []
for order_idx, feat_name in enumerate(expected_features, start=1):
    dt_str = str(selected_df[feat_name].dtype)

    if feat_name.startswith('ohe_'):
        feat_cat = 'categorical_one_hot'
        prefix_matched = [p for p in ['ohe_employment_status_', 'ohe_residence_status_', 'ohe_educational_attainment_', 'ohe_diagnosis_severity_category_', 'ohe_type_of_treatment_surgery_', 'ohe_assistance_type_'] if feat_name.startswith(p)]
        raw_prefix = prefix_matched[0] if prefix_matched else 'ohe_'
        src_cols = raw_prefix.replace('ohe_', '').rstrip('_')
        xf_desc = 'SimpleImputer + OneHotEncoder(handle_unknown=ignore)'
        mv_policy = 'Missing values imputed with training-partition mode prior to one-hot encoding (0 post-imputation nulls).'
        fair_note = 'Employment status asymmetry noted in dataset audit; evaluated alongside protected attributes in post-modeling fairness audit.' if src_cols == 'employment_status' else 'Standard pre-decision socioeconomic/clinical category.'
    else:
        base_col = dist_raw_name_map.get(feat_name, feat_name)
        src_cols = fe_source_map.get(base_col, base_col)
        xf_desc = dist_method_map.get(feat_name, fe_formula_map.get(base_col, 'Passthrough_PreDecision'))
        if 'income' in base_col or 'obligation' in base_col or 'margin' in base_col or 'household' in base_col or 'dependent' in base_col or 'utility' in base_col:
            feat_cat = 'socioeconomic_numerical'
        elif 'documentation' in base_col:
            feat_cat = 'clinical_document_availability'
        else:
            feat_cat = 'clinical_cost_or_severity'
        mv_policy = (
            'Binary 0/1 indicator with 0 structural nulls.'
            if xf_desc == 'Passthrough_Binary_0_1'
            else 'Preserved NaN during feature creation; imputed strictly with training-partition median (n=350) inside transformation pipeline.'
        )
        fair_note = 'Non-protected pre-decision predictor; demographic attributes (sex, age) isolated separately.'

    manifest_rows.append({
        'feature_order': order_idx,
        'feature_name': feat_name,
        'data_type': dt_str,
        'feature_category': feat_cat,
        'source_columns': src_cols,
        'transformation': xf_desc,
        'model_input_eligibility': 'ELIGIBLE_PRIMARY_INPUT',
        'leakage_status': 'VERIFIED_PRE_DECISION_NO_LEAKAGE',
        'fairness_consideration': fair_note,
        'missing_value_policy': mv_policy,
    })

manifest_df = pd.DataFrame(manifest_rows)
print(f'Constructed final feature manifest for {len(manifest_df)} validated features:')
print(manifest_df[['feature_order', 'feature_name', 'data_type', 'feature_category', 'source_columns']].head(20).to_string(index=False))

Constructed final feature manifest for 41 validated features:
 feature_order                                         feature_name data_type               feature_category                                                           source_columns
             1                 xf_log_rob_per_capita_monthly_income   float64        socioeconomic_numerical                                    monthly_family_income; household_size
             2                  xf_log_rob_monthly_utility_expenses   float64        socioeconomic_numerical                                                 monthly_utility_expenses
             3                         xf_log_rob_cost_of_treatment   float64      clinical_cost_or_severity                                                        cost_of_treatment
             4                xf_log_rob_income_to_obligation_ratio   float64        socioeconomic_numerical            monthly_family_income; monthly_household_obligations_expenses
             5              

### 5. Export Final Feature Schema JSON, Feature Manifest CSV & Validation Report CSV

In [5]:
manifest_df.to_csv(FINAL_MANIFEST_CSV_OUT, index=False)
validation_df.to_csv(VALIDATION_CSV_OUT, index=False)

schema_hash = hashlib.sha256('|'.join(expected_features).encode('utf-8')).hexdigest()
all_checks_passed = bool((validation_df['status'] == 'PASS').all())

final_schema_payload = {
    'schema_version': '2.0.0',
    'notebook': '03_feature_engineering/04_feature_schema_validation.ipynb',
    'generated_at_utc': datetime.datetime.now(datetime.timezone.utc).isoformat(),
    'readiness_status': 'READY_WITH_DOCUMENTED_LIMITATIONS' if all_checks_passed else 'NOT_READY',
    'all_mandatory_validation_checks_passed': all_checks_passed,
    'random_seed': RANDOM_SEED,
    'raw_dataset_path': 'data/synthetic/tabular/ofphil_synthetic_raw.csv',
    'raw_dataset_sha256': raw_sha256,
    'selected_features_dataset_path': 'data/interim/features/selected_features.csv',
    'fairness_demographics_dataset_path': 'data/interim/features/fairness_demographics_dataset.csv',
    'target_specification': {
        'column_name': 'eligibility',
        'data_type': 'str (categorical binary target)',
        'allowed_values': ['ELIGIBLE', 'INELIGIBLE'],
        'class_distribution': target_counts,
        'prevalence_limitation_note': '50/50 balanced cohort (522 ELIGIBLE [50.0%] vs 522 INELIGIBLE [50.0%], N=1,044) combining 44 adjudicated original intake cases and 1,000 calibrated synthetic cases; 6 PENDING original cases quarantined.',
    },
    'partition_counts': {
        'train': int(len(X_train)),
        'validation': int(len(X_val)),
        'test': int(len(X_test)),
        'total': int(len(selected_df)),
    },
    'feature_count': int(len(expected_features)),
    'feature_Order_sha256': schema_hash,
    'features': manifest_rows,
    'enforced_leakage_exclusions': selection_cfg['mandatory_leakage_exclusions'],
    'high_risk_proxy_review_exclusions': selection_cfg['high_risk_proxy_review_exclusions'],
    'isolated_protected_fairness_attributes': selection_cfg['isolated_protected_fairness_attributes'],
    'validation_summary': {
        'total_checks_executed': int(len(validation_df)),
        'checks_passed': int((validation_df['status'] == 'PASS').sum()),
        'checks_failed': int((validation_df['status'] != 'PASS').sum()),
    },
}
FINAL_SCHEMA_JSON_OUT.write_text(json.dumps(final_schema_payload, indent=2), encoding='utf-8')

print(f'Saved final feature schema JSON -> {FINAL_SCHEMA_JSON_OUT}')
print(f'Saved final feature manifest CSV -> {FINAL_MANIFEST_CSV_OUT} ({len(manifest_df)} features)')
print(f'Saved feature schema validation report CSV -> {VALIDATION_CSV_OUT} ({len(validation_df)} checks)')
print(f'Final Schema Readiness Verdict: {final_schema_payload["readiness_status"]}')

Saved final feature schema JSON -> /home/kristoper/Projects/Thesis/ofphil-classification-v2/data/processed/features/final_feature_schema.json
Saved final feature manifest CSV -> /home/kristoper/Projects/Thesis/ofphil-classification-v2/data/processed/features/final_feature_manifest.csv (41 features)
Saved feature schema validation report CSV -> /home/kristoper/Projects/Thesis/ofphil-classification-v2/data/processed/validation/feature_schema_validation.csv (15 checks)
Final Schema Readiness Verdict: READY_WITH_DOCUMENTED_LIMITATIONS


### Interpretation & Limitations

1. **Verified Alignment with Actual V2 Schema:** The feature contract uses the actual target column `eligibility` (`ELIGIBLE` / `INELIGIBLE`) and traces every selected predictor to its exact raw source column(s) in `ofphil_synthetic_data_dictionary.csv`.
2. **Decision-Support Governance:** The validated feature matrix is designed strictly for institutional decision-support research. Model outputs must never be treated as automatic patient approval or rejection, nor as a replacement for licensed Medical Social Workers.
3. **Documented Synthetic Dataset Limitations:**
   - The 50/50 target distribution is experimental.
   - Multi-item Stage-2 MSW rubric sub-scores (`Average Severity`, `Average Coping`) are absent from the 59-column raw schema.
   - Socio-legal document-count proxies (`total_supporting_documents` and `certificate_of_indigency`) are excluded from the primary feature contract to prevent synthetic proxy shortcuts.


### Completion Checklist

- [x] Verified target column is `eligibility` and removed all legacy placeholder column names.
- [x] Executed 15 programmatic validation checks covering columns, dtypes, nulls, infinities, duplicates, ordering, leakage exclusions, fairness isolation, and cross-split consistency.
- [x] Exported `data/processed/features/final_feature_schema.json`.
- [x] Exported `data/processed/features/final_feature_manifest.csv`.
- [x] Exported `data/processed/validation/feature_schema_validation.csv`.
